# Preprocess code 1 - A1 retailer
## Universidad ICESI 
### David Mauricio Orozco Rios
### author: Davoroz06 - IG

In [ ]:
# libraries
import pandas as pd
import numpy as np

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text

Load data:

In [ ]:
data = pd.read_csv(wd_dp + "A1_raw.csv")
len(data)

Missing description

In [ ]:
data = data[~data['Descripcion'].isna()]
len(data)

String homogenize

In [ ]:
data["precio"] = data['precio'].fillna(data['low'])
data['palabra'] = data['palabra'].fillna(data['Link'].str.extract(r"https?://[^/]+/(.*)\?").squeeze())
data["palabra"] = data["palabra"].apply(homogenize_text)
data["descripcion"] = data["Descripcion"].apply(homogenize_text)

Price homogenize

In [ ]:
data["precio"] = data["precio"].astype("str")
data["precio"] = data["precio"].str.replace("\.0$", "", regex=True)
data["precio"] = data["precio"].str.replace("[a-zA-Z]", "", regex=True)
data["precio"] = data["precio"].str.replace(".", "")
data["precio"] = data["precio"].str.replace(",", ".")

Filter empty price products

In [ ]:
data = data[data["precio"].str.len()!=0]
data = data[~data['precio'].isna()]

Price to numeric

In [ ]:
data[data["descripcion"].str.contains("adorno")]["precio"].unique()

In [ ]:
data["precio"] = data["precio"].astype("float")

Unit price homogenize

In [ ]:
data["precio_und"] = data["precio_und"].astype("str")
data["p_und"] = data["precio_und"].str.replace("[^0-9\,]", "", regex=True)
data["p_und"] = data["p_und"].str.replace(",", ".", regex=False)
data["p_und"] = data["p_und"].replace("", np.nan)
data["p_und"] = data["p_und"].astype("float")

Unit price homogenize

In [ ]:
data["und"] = data["precio_und"].str.extract("([A-Za-z]+)", expand = True)

Size

In [ ]:
data['tamano'] = data['tamano'].fillna(data['precio']/data['p_und'])

Link

In [ ]:
data['link'] = data['link'].fillna(data['Link'])

Date

In [ ]:
data['fecha'] = data['fecha'].fillna(data['Fecha'])

Create quantities

In [ ]:
data["unidad_liquida"] = data["descripcion"].str.extract(r"([0-9]+ ml|[0-9]+ml|[0-9]+ mililitro)")
data["unidad_liquida"] = data["unidad_liquida"].str.replace(r"[\s]|[mililitro]", "", regex = True)
data["cantidad"] = data["descripcion"].str.extract(r"([0-9]+und|[0-9]+ und|x[\s]*[0-9]+|[0-9]+[\s]*x)")
data["cantidad"] = data["cantidad"].str.replace(r"([a-z\s]+)", "", regex = True)
#data["p_und"] = data["p_und"].replace("", np.nan)
data.loc[data['descripcion'].str.contains("sixpack"), 'cantidad'] = 6
data.loc[data['descripcion'].str.contains("fourpack"), 'cantidad'] = 4
data.loc[(data['palabra'].str.contains("cerveza"))&(data["cantidad"].isna()), 'cantidad'] = 1

Quantities to numeric

In [ ]:
data["unidad_liquida"] = data["unidad_liquida"].astype(float)
data["cantidad"] = data["cantidad"].replace("", np.nan)
data["cantidad"] = data["cantidad"].astype(float)

Price per liquid quantities

In [ ]:
data["p_und_liquida"] = data["precio"]/data["unidad_liquida"]

Select columns

In [ ]:
data = data[["fecha", "descripcion", "precio", "palabra"]]
data["tienda"] = "A1"

Summarize duplicate prices

In [ ]:
data = data.groupby(['fecha', 'descripcion', 'tienda']).agg({'precio': 'min'}).reset_index()

Save preprocess data

In [ ]:
pd.DataFrame(data.to_csv(wd_db+"A1_clean.csv"))